
# Hellish Conditions for Seixas?

In February 2026, the Decathlon CMA CGM cycling team was at a training camp in southern Spain.

On February 14, after returning from a training session, rising French cycling star Paul Seixas posted the ride on his public Strava account with the comment: "A bit dangerous this wind 😬"

![Strava Paul Seixas](/_static/Stravaseixas.png)

I immediately downloaded the GPX and measured the wind conditions along his ride...

In [7]:
from datetime import timedelta, datetime
from zoneinfo import ZoneInfo

from refwindcycle.core import Simulator
from refwindcycle.core.cyclist_params import create_pro_profile

from refwindcycle.weather import WeatherProvider
from refwindcycle.weather.grib_finder import build_grib_list
from refwindcycle.analysis import RouteAnalyzer
from refwindcycle.analysis.anareswind import print_summary_statistics

## 1. Building the wind model


We cannot use the time provided by the GPX file because it does not contain any timestamp. GPX files downloaded from Strava do not include timestamps unless it is a file associated with your account.

Since the GPX file does not contain timestamps, a replay simulation is not possible.

However, it is still possible to run a simulation in "future mode" by choosing the actual date and time of the ride.

This will allow us to assess the actual wind conditions encountered by Paul Seixas.

Therefore, we will use the date displayed on the Strava webpage.

In [8]:
# Use the Strava start time
local_tz = ZoneInfo("Europe/Madrid")
departure_time = '2026-02-14 10:25:00'
t_start = datetime.fromisoformat(departure_time).replace(tzinfo=local_tz)

# Ride duration
estimated_ride_duration_h = 4

# Define the directory used to store GRIB files
hdir = "~/data"  # GRIB files stored in ~/data by default

gribs_list = build_grib_list(hdir, t_start, 1, estimated_ride_duration_h)
weather = WeatherProvider(gribs_list)
mygrib = weather.grib
print("Grib loaded")

Grib loaded


## 2. Parameters and simulator initialization

We use typical pro-level parameters for this simulation. Paul Seixas's weight is easy to find online.

We previously ran several simulations to tune a base power that gives realistic results (time and average speed). For this tuning, we selected `v0` values consistent with the average speed reported by Strava and the elevation profile.

Remember that `v0` is the speed corresponding to power `P0` on flat terrain and without wind. Here, the route has significant elevation gain (> 2000 m) and strong wind. A `v0` of 39 km/h is a realistic choice to obtain an average speed close to 32 km/h.


In [9]:
pro = create_pro_profile()
procda = 0.28
procr = 0.0035
promass = 71

sim = Simulator(mygrib, behavior=pro, CdA=procda, Cr=procr, m=promass)

v0 = 39 / 3.6  # m/s
P0 = sim.P0_from_v0(v0)

print(f"Input parameters for the simulation: v0={v0*3.6:.2f} km/h, P0={P0:.1f} W, CdA={procda}, Cr={procr}, m={promass}")

Input parameters for the simulation: v0=39.00 km/h, P0=244.4 W, CdA=0.28, Cr=0.0035, m=71


## 3. Loading the GPX

GPX preprocessing. Stop filtering is not relevant here because this GPX has no timestamps.

In [10]:

filegpx="./P20260214-103K-1025_Seixas.gpx"
analyzer = RouteAnalyzer()
segments, stats = analyzer.process_gpx(filegpx, verbose=True)




  NETTOYAGE GPX: ./P20260214-103K-1025_Seixas.gpx
[1/6] Chargement du fichier GPX...
[2/6] Création des segments (smooth=True)...
      11467 segments créés, denivelé: 2350.4 m
[3/6] Détection des bruits GPS...
      ⚠ 6 segments aberrants détectés
      Suppression des bruits...
[4/6] Fusion des segments courts (min_dist=50.0m)...
      5606 segments fusionnés
[5/6] Filtrage arrêts: désactivé
[6/6] Résumé du traitement:
      Segments: 11467 -> 5855
      Distance: 102.996km -> 102.973km
      Denivelé: 2350.4m -> 2305.5m
      Rectangle SN: (36, 37), EW: (-5, -4)



## 4. Simulation launch

Before running the simulation, we recall the Strava ride summary:

**Distance**: 103.20 km   **Time**: 3:13:54   **Elevation gain**: 2416 m

**Average**: 31.9 km/h   **Max**: 87.3 km/h

The simulation results are very close.

I cannot guarantee that the estimated power exactly matches the value measured by Paul Seixas's power meter, but it seems realistic for a training ride.


In [11]:


results = sim.simulate_future(segments, t_start, P0=P0)
print("\n--- Simulation du parcours avec le vent le " + str(t_start) + " ---")
print(f"\nAverage speed: {results.speed.avg:.2f} km/h - Moving Average speed: {results.speed.moving_avg:.2f} km/h - Max speed: {results.speed.max:.2f} km/h")
print(f"Average power: {results.power.avg:.1f} W" if results.power else "Average power: n/a")
print(f"temps= {str(timedelta(seconds=int(results.time.total_seconds)))} , longueur= {results.distance.total_km:.2f} km")


--- Simulation du parcours avec le vent le 2026-02-14 10:25:00+01:00 ---

Average speed: 31.81 km/h - Moving Average speed: 31.81 km/h - Max speed: 88.05 km/h
Average power: 270.0 W
temps= 3:14:15 , longueur= 102.97 km


## 4. Wind analysis

The ride summary gives a WindScore grade of `F`.

This grade is driven by both safety and performance criteria.

In [12]:
print_summary_statistics(results, "Parcours de Seixas du 14 février 2026 avec vent")




  STATISTIQUES - Parcours de Seixas du 14 février 2026 avec vent

📏 Distance totale: 102.97 km
⏱️  Temps total: 03:14:15
🚴  Vitesse moyenne: 31.81 km/h (max=88.05 km/h)
⚡  Puissance moyenne: 270.0 W

💨 Vent (TWS et TWD):
   Moyen: 21.79 km/h - Direction: 327° (NNW)
   Min: 15.12 km/h
   Max: 25.81 km/h

💨 Rafales (Gust):
   Moyenne: 32.19 km/h
   Min: 26.86 km/h (au km 83.00)
   Max: 38.20 km/h (au km 0.23)

⛰️ Pente terrain:
   Moyenne: 2.24 %
   Min (lissé 100m): -16.28% (au km 71.97)
   Max (lissé 100m): 14.98% (au km 65.41)
   Dénivelé positif: 2306 m
   Dénivelé négatif: -2346 m

🌬️ Pente virtuelle (vent):
   Moyenne: 0.26%
   Min (lissé 100m): -8.69% (au km 100.77)
   Max (lissé 100m): 6.86% (au km 15.60)
   Dénivelé virtuel positif: 1222 m
   Dénivelé virtuel négatif: -1550 m

📊 Pente effective (terrain + vent):
   Moyenne: 2.49%
   Min (lissé 100m): -20.19% (au km 95.03)
   Max (lissé 100m): 15.77% (au km 65.42)
   Dénivelé effectif positif: 3528 m
   Dénivelé effectif négatif

## 5. Conclusion

The main reasons for this poor safety rating are:

- strong gusts,

- high average wind speed (TWS),

- high lateral instability (crosswind).

Paul Seixas was right to highlight how dangerous the wind was. Beyond his impressive performance for his age, this also shows strong maturity and risk awareness.



In [14]:
print(f"Safety danger score of {results.wind_score.safety_danger_score} out of a maximum of 7\nReasons:")

gust_max_kmh = results.gusts.max_kmh
print(f" Gust max: {gust_max_kmh:.2f} km/h")

crosswind_avg_kmh = results.crosswind.avg_kmh
print(f" Crosswind average: {crosswind_avg_kmh:.2f} km/h")

wind_avg_kmh = results.wind.tws.avg_kmh
print(f" TWS average: {wind_avg_kmh:.2f} km/h")

Safety danger score of 7 out of a maximum of 7
Reasons:
 Gust max: 38.20 km/h
 Crosswind average: 14.07 km/h
 TWS average: 21.79 km/h
